## 1. Import Libraries & Setup

In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform, randint, uniform

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, fbeta_score,
    recall_score, precision_score, f1_score,
    precision_recall_curve, average_precision_score, roc_auc_score,
    matthews_corrcoef, make_scorer
)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X_train shape: (8000, 9) | X_test shape: (2000, 9)
Target distribution in Train (%):
machine_failure
0    96.61
1     3.39
Target distribution in Test (%):
machine_failure
0    96.6
1     3.4


## 2. Load Processed Datasets

In [2]:
data_dir = '../data/processed'
if not os.path.exists(data_dir):
    data_dir = 'data/processed'

X_train = pd.read_csv(os.path.join(data_dir, 'X_train.csv'))
X_test  = pd.read_csv(os.path.join(data_dir, 'X_test.csv'))
y_train_full = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
y_test_full  = pd.read_csv(os.path.join(data_dir, 'y_test.csv'))

y_train = y_train_full['machine_failure']
y_test  = y_test_full['machine_failure']

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print("Target distribution in Train (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("Target distribution in Test (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

           Model  Best CV F2
0       LightGBM      0.8276
1  Random Forest      0.8146


## 3. Train-Validation Split for Selection & Evaluation Routine

In [3]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
)

scaler = StandardScaler()
X_tr_scaled  = pd.DataFrame(scaler.fit_transform(X_tr),  columns=X_tr.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val),     columns=X_val.columns)
X_train_scaled_full = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled       = pd.DataFrame(scaler.transform(X_test),       columns=X_test.columns)

def f2_score_func(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)

f2_scorer = make_scorer(f2_score_func)

def evaluate_model(name, y_true, y_pred, y_proba):
    return {
        'model': name,
        'recall': recall_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred),
        'f2': fbeta_score(y_true, y_pred, beta=2),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'pr_auc': average_precision_score(y_true, y_proba),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

               recall  precision      f1      f2     mcc  pr_auc  roc_auc  threshold
model                                                                               
LightGBM       0.7794     0.7910  0.7852  0.7817  0.7777  0.8383   0.9799     0.7107
Random Forest  0.8235     0.6222  0.7089  0.7735  0.7046  0.8496   0.9794     0.3297
Selected Winner Model: LightGBM with Optimal Threshold: 0.7107


## 4. Hyperparameter Tuning (RandomizedSearchCV)

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scale_pos_weight_val = (y_tr == 0).sum() / (y_tr == 1).sum()

param_grids = {
    'Logistic Regression': {
        'model': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE),
        'params': {'C': loguniform(1e-3, 1e2), 'penalty': ['l2']},
        'use_scaled': True
    },
    'Random Forest': {
        'model': RandomForestClassifier(class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'n_estimators': randint(100, 400),
            'max_depth': [None, 8, 12, 16],
            'min_samples_split': randint(2, 10),
            'min_samples_leaf': randint(1, 5)
        },
        'use_scaled': False
    },
    'XGBoost': {
        'model': XGBClassifier(scale_pos_weight=scale_pos_weight_val, eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'n_estimators': randint(100, 400),
            'max_depth': randint(3, 8),
            'learning_rate': loguniform(0.01, 0.2),
            'subsample': uniform(0.6, 0.4),
            'colsample_bytree': uniform(0.6, 0.4)
        },
        'use_scaled': False
    },
    'LightGBM': {
        'model': LGBMClassifier(scale_pos_weight=scale_pos_weight_val, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        'params': {
            'n_estimators': randint(100, 400),
            'max_depth': randint(3, 8),
            'learning_rate': loguniform(0.01, 0.2),
            'num_leaves': randint(15, 63),
            'subsample': uniform(0.6, 0.4),
            'colsample_bytree': uniform(0.6, 0.4)
        },
        'use_scaled': False
    }
}

tuned_models_val = {}
tuning_summary = []

for name, cfg in param_grids.items():
    X_data = X_tr_scaled if cfg['use_scaled'] else X_tr
    search = RandomizedSearchCV(
        cfg['model'], cfg['params'], n_iter=20, scoring=f2_scorer,
        cv=cv, random_state=RANDOM_STATE, n_jobs=-1
    )
    search.fit(X_data, y_tr)
    tuned_models_val[name] = {
        'best_estimator': search.best_estimator_,
        'best_params': search.best_params_,
        'best_cv_f2': search.best_score_,
        'use_scaled': cfg['use_scaled']
    }
    tuning_summary.append({
        'Model': name,
        'Best CV F2': round(search.best_score_, 4),
        'Best Params': search.best_params_
    })

display(pd.DataFrame(tuning_summary))

                       recall  precision      f1     f2    mcc  pr_auc  roc_auc  threshold
model                                                                                     
LightGBM (Final Test)  0.8529     0.8286  0.8406  0.848  0.835  0.8967   0.9853     0.7107


## 5. Model Selection & Threshold Optimization

In [5]:
val_results = []
threshold_data = {}

for name, info in tuned_models_val.items():
    model_obj = info['best_estimator']
    X_val_data = X_val_scaled if info['use_scaled'] else X_val
    y_proba_val = model_obj.predict_proba(X_val_data)[:, 1]
    
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba_val)
    f2_scores = (5 * precisions[:-1] * recalls[:-1]) / (4 * precisions[:-1] + recalls[:-1] + 1e-10)
    best_idx = np.argmax(f2_scores)
    best_thresh = thresholds[best_idx]
    
    y_pred_val = (y_proba_val >= best_thresh).astype(int)
    res = evaluate_model(name, y_val, y_pred_val, y_proba_val)
    res['threshold'] = best_thresh
    val_results.append(res)
    threshold_data[name] = best_thresh

val_comparison_df = pd.DataFrame(val_results).set_index('model')
display(val_comparison_df.round(4))

winner_name = val_comparison_df['f2'].idxmax()
winner_info = tuned_models_val[winner_name]
best_threshold = threshold_data[winner_name]
winner_model_val = winner_info['best_estimator']

print(f"Selected Winner Model: {winner_name} with Optimal Threshold: {best_threshold:.4f}")

Best model (LightGBM) and threshold saved to models/


## 6. Retrain Winner on Full Training Set & Test Evaluation

In [6]:
scale_pos_weight_full_tr = (y_train == 0).sum() / (y_train == 1).sum()
winner_params = winner_info['best_params']

model_constructors = {
    'Logistic Regression': lambda: LogisticRegression(**winner_params, class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE),
    'Random Forest': lambda: RandomForestClassifier(**winner_params, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': lambda: XGBClassifier(**winner_params, scale_pos_weight=scale_pos_weight_full_tr, eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM': lambda: LGBMClassifier(**winner_params, scale_pos_weight=scale_pos_weight_full_tr, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

best_model = model_constructors[winner_name]()

if winner_info['use_scaled']:
    best_model.fit(X_train_scaled_full, y_train)
    y_proba_test = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    best_model.fit(X_train, y_train)
    y_proba_test = best_model.predict_proba(X_test)[:, 1]

y_pred_test = (y_proba_test >= best_threshold).astype(int)
final_results = evaluate_model(f'{winner_name} (Final Test)', y_test, y_pred_test, y_proba_test)
final_results['threshold'] = best_threshold

display(pd.DataFrame([final_results]).set_index('model').round(4))

## 7. Model Export

In [7]:
model_export_dir = '../models'
if not os.path.exists(model_export_dir):
    model_export_dir = 'models'
os.makedirs(model_export_dir, exist_ok=True)

joblib.dump(best_model, os.path.join(model_export_dir, 'best_model.pkl'))
joblib.dump(best_threshold, os.path.join(model_export_dir, 'best_threshold.pkl'))
if winner_info['use_scaled']:
    joblib.dump(scaler_full, os.path.join(model_export_dir, 'scaler.pkl'))

print(f"Best model ({winner_name}) and threshold saved to {model_export_dir}/")